# 03 — Gold Layer: Business Aggregations & Exploration

**Purpose:** Read from Silver and produce business-ready aggregated tables.  
Then explore the results to understand what the data tells us.

**Medallion layer:** 🥇 Gold — analytics-ready, one table per business question

**What you will learn in this notebook:**
- How to design Gold tables around business questions
- How to use `groupBy().agg()` for multi-metric aggregations
- How to use window functions for ranking and running totals
- How to use `create_map()` for categorical labeling
- How to write multiple Gold Delta tables
- How to query Gold tables for business insights

---
**Input:**  Silver Delta table (`data/silver/`)  
**Output:** 5 Gold Delta tables in `data/gold/`  

| Gold Table | Business Question |
|---|---|
| `daily_revenue` | How much revenue per day? |
| `hourly_patterns` | When is demand highest? |
| `payment_analysis` | How do customers pay? |
| `top_routes` | Which routes are most popular? |
| `distance_bands` | Short hops vs long journeys? |

## 0. Setup

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_PATH = os.path.join(PROJECT_ROOT, 'data', 'silver')
GOLD_PATH   = os.path.join(PROJECT_ROOT, 'data', 'gold')

os.makedirs(GOLD_PATH, exist_ok=True)
print(f"Silver : {SILVER_PATH}")
print(f"Gold   : {GOLD_PATH}")

In [ ]:
spark = (
    SparkSession.builder
    .appName("nyc-taxi-gold")
    .master("local[*]")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready")

## 1. Read Silver

We exclude outlier rows from Gold — they would skew averages.  
Outliers are preserved in Silver for audit purposes.

In [ ]:
df = (
    spark.read.format("delta").load(SILVER_PATH)
    .filter(F.col("is_outlier") == False)   # exclude fare > $500
)

total = df.count()
print(f"Silver rows (outliers excluded): {total:,}")

# Cache Silver — we'll query it 5 times (one per Gold table)
# Caching prevents re-reading from disk each time
df.cache()
print("Silver DataFrame cached in memory")

## 2. Gold Table 1 — Daily Revenue

**Business question:** How much revenue did we generate each day?  
**Used by:** Finance dashboards, daily reporting, trend analysis

In [ ]:
daily_revenue = (
    df
    .groupBy("pickup_date")
    .agg(
        F.count("*")                              .alias("total_trips"),
        F.round(F.sum("fare_amount"),         2)  .alias("total_fare_revenue"),
        F.round(F.sum("total_amount"),        2)  .alias("total_revenue"),
        F.round(F.sum("tip_amount"),          2)  .alias("total_tips"),
        F.round(F.avg("fare_amount"),         2)  .alias("avg_fare"),
        F.round(F.avg("trip_distance"),       2)  .alias("avg_distance_miles"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
        F.round(F.avg("passenger_count"),     2)  .alias("avg_passengers"),
    )
    .orderBy("pickup_date")
)

print(f"Daily revenue rows: {daily_revenue.count()}")
daily_revenue.show(10)

In [ ]:
# Add 7-day rolling average revenue — useful for smoothing daily noise
# This is an advanced window function pattern interviewers love
window_7d = (
    Window
    .orderBy("pickup_date")
    .rowsBetween(-6, 0)   # current row + 6 preceding = 7-day window
)

daily_revenue = daily_revenue.withColumn(
    "revenue_7day_avg",
    F.round(F.avg("total_revenue").over(window_7d), 2)
)

print("With 7-day rolling average:")
daily_revenue.select(
    "pickup_date", "total_trips", "total_revenue", "revenue_7day_avg"
).show(14)

In [ ]:
# Write Gold table 1
daily_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(os.path.join(GOLD_PATH, "daily_revenue"))

print(f"✓ daily_revenue written → {GOLD_PATH}/daily_revenue")

## 3. Gold Table 2 — Hourly Demand Patterns

**Business question:** When is demand highest during the day and week?  
**Used by:** Driver positioning, surge pricing, staffing decisions

In [ ]:
dow_labels = F.create_map(
    F.lit(1), F.lit("Sunday"),    F.lit(2), F.lit("Monday"),
    F.lit(3), F.lit("Tuesday"),   F.lit(4), F.lit("Wednesday"),
    F.lit(5), F.lit("Thursday"),  F.lit(6), F.lit("Friday"),
    F.lit(7), F.lit("Saturday"),
)

hourly_patterns = (
    df
    .groupBy("pickup_hour", "pickup_dow")
    .agg(
        F.count("*")                              .alias("total_trips"),
        F.round(F.avg("fare_amount"),         2)  .alias("avg_fare"),
        F.round(F.avg("trip_distance"),       2)  .alias("avg_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
    )
    .withColumn("day_name", dow_labels[F.col("pickup_dow")])
    .orderBy("pickup_dow", "pickup_hour")
)

print(f"Hourly pattern rows: {hourly_patterns.count()}  (7 days × 24 hours = 168 max)")
hourly_patterns.show(10)

In [ ]:
# Find the top 5 busiest hour/day combinations
print("Top 5 busiest hour + day combinations:")
hourly_patterns.orderBy(F.col("total_trips").desc()).show(5)

In [ ]:
hourly_patterns.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(os.path.join(GOLD_PATH, "hourly_patterns"))

print(f"✓ hourly_patterns written")

## 4. Gold Table 3 — Payment Analysis

**Business question:** How do customers pay, and how does it affect tipping?  
**Used by:** Revenue analysis, tip optimisation, UX decisions

In [ ]:
payment_analysis = (
    df
    .groupBy("payment_type_label")
    .agg(
        F.count("*")                        .alias("trip_count"),
        F.round(F.sum("total_amount"),  2)  .alias("total_revenue"),
        F.round(F.avg("total_amount"),  2)  .alias("avg_fare"),
        F.round(F.avg("tip_amount"),    2)  .alias("avg_tip"),
        F.round(
            F.sum("tip_amount") / F.sum("fare_amount") * 100, 2
        )                                   .alias("tip_rate_pct"),
    )
    .withColumn(
        "share_of_trips_pct",
        F.round(F.col("trip_count") / total * 100, 2)
    )
    .orderBy(F.col("trip_count").desc())
)

payment_analysis.show(truncate=False)

In [ ]:
payment_analysis.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(os.path.join(GOLD_PATH, "payment_analysis"))

print("✓ payment_analysis written")

## 5. Gold Table 4 — Top Routes

**Business question:** Which pickup → dropoff zone pairs are most popular?  
**Used by:** Fleet positioning, route optimisation, advertising

In [ ]:
top_routes = (
    df
    .groupBy("pickup_location_id", "dropoff_location_id")
    .agg(
        F.count("*")                        .alias("trip_count"),
        F.round(F.avg("fare_amount"),   2)  .alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2)  .alias("avg_distance"),
        F.round(F.avg("tip_amount"),    2)  .alias("avg_tip"),
    )
    .orderBy(F.col("trip_count").desc())
    .limit(50)
)

print("Top 10 routes by trip count:")
top_routes.show(10)

In [ ]:
# Advanced: rank routes within each pickup zone using window functions
# Shows: for each pickup zone, what is the most common destination?
window_rank = Window.partitionBy("pickup_location_id").orderBy(
    F.col("trip_count").desc()
)

top_routes_ranked = top_routes.withColumn(
    "rank_within_pickup_zone",
    F.rank().over(window_rank)
)

# Show top destination for the 5 busiest pickup zones
print("Top destination per pickup zone (rank=1 only):")
top_routes_ranked.filter(F.col("rank_within_pickup_zone") == 1) \
                 .orderBy(F.col("trip_count").desc()) \
                 .show(5)

In [ ]:
top_routes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(os.path.join(GOLD_PATH, "top_routes"))

print("✓ top_routes written")

## 6. Gold Table 5 — Distance Bands

**Business question:** What proportion of trips are short hops vs long journeys?  
**Used by:** Pricing strategy, driver incentives, service zone planning

In [ ]:
distance_bands = (
    df
    .withColumn(
        "distance_band",
        F.when(F.col("trip_distance") < 1,  "< 1 mile")
         .when(F.col("trip_distance") < 3,  "1–3 miles")
         .when(F.col("trip_distance") < 7,  "3–7 miles")
         .when(F.col("trip_distance") < 15, "7–15 miles")
         .otherwise("15+ miles")
    )
    .groupBy("distance_band")
    .agg(
        F.count("*")                              .alias("trip_count"),
        F.round(F.avg("fare_amount"),         2)  .alias("avg_fare"),
        F.round(F.avg("tip_amount"),          2)  .alias("avg_tip"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
    )
    .withColumn(
        "share_pct",
        F.round(F.col("trip_count") / total * 100, 2)
    )
    .orderBy("distance_band")
)

distance_bands.show()

In [ ]:
distance_bands.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(os.path.join(GOLD_PATH, "distance_bands"))

print("✓ distance_bands written")

## 7. Gold Data Quality Checks

In [ ]:
df_daily = spark.read.format("delta").load(
    os.path.join(GOLD_PATH, "daily_revenue")
)

total_rev  = df_daily.agg(F.sum("total_revenue")).collect()[0][0] or 0
avg_fare   = df_daily.agg(F.avg("avg_fare")).collect()[0][0] or 0
daily_rows = df_daily.count()

checks = [
    ("daily_revenue has rows",         daily_rows > 0),
    ("total revenue is positive",      total_rev > 0),
    ("avg fare > $2 (not too low)",    avg_fare > 2),
    ("avg fare < $200 (not too high)", avg_fare < 200),
]

print("Gold Data Quality Report")
print("-" * 45)
for name, passed in checks:
    print(f"  {'✓' if passed else '✗'}  {name}")
print("-" * 45)
print(f"  Total revenue : ${total_rev:,.2f}")
print(f"  Avg fare      : ${avg_fare:.2f}")
print(f"  Days in table : {daily_rows}")

## 8. Business Insights Summary

Run this cell last — it prints a business-friendly summary of the month.

In [ ]:
# Pull summary metrics from Gold tables
df_dr  = spark.read.format("delta").load(os.path.join(GOLD_PATH, "daily_revenue"))
df_pay = spark.read.format("delta").load(os.path.join(GOLD_PATH, "payment_analysis"))
df_hr  = spark.read.format("delta").load(os.path.join(GOLD_PATH, "hourly_patterns"))
df_db  = spark.read.format("delta").load(os.path.join(GOLD_PATH, "distance_bands"))

total_trips    = df_dr.agg(F.sum("total_trips")).collect()[0][0]
total_revenue  = df_dr.agg(F.sum("total_revenue")).collect()[0][0]
avg_daily_rev  = df_dr.agg(F.avg("total_revenue")).collect()[0][0]
avg_fare_val   = df_dr.agg(F.avg("avg_fare")).collect()[0][0]
avg_dist_val   = df_dr.agg(F.avg("avg_distance_miles")).collect()[0][0]

top_payment    = df_pay.orderBy(F.col("trip_count").desc()).first()["payment_type_label"]
top_payment_pct = df_pay.orderBy(F.col("trip_count").desc()).first()["share_of_trips_pct"]

peak_hour      = df_hr.orderBy(F.col("total_trips").desc()).first()["pickup_hour"]
peak_day       = df_hr.orderBy(F.col("total_trips").desc()).first()["day_name"]

short_trips    = df_db.filter(F.col("distance_band") == "< 1 mile").first()
short_pct      = short_trips["share_pct"] if short_trips else 0

print(f"""
╔═══════════════════════════════════════════════════════════╗
║         NYC YELLOW TAXI — January 2024 Summary            ║
╠═══════════════════════════════════════════════════════════╣
║  Total trips       : {total_trips:>12,.0f}                    ║
║  Total revenue     : ${total_revenue:>12,.2f}                  ║
║  Avg daily revenue : ${avg_daily_rev:>12,.2f}                  ║
║  Avg fare          : ${avg_fare_val:>12.2f}                  ║
║  Avg trip distance : {avg_dist_val:>11.2f} mi                 ║
╠═══════════════════════════════════════════════════════════╣
║  Top payment method: {top_payment:<20} ({top_payment_pct}%)  ║
║  Peak demand       : {peak_day:<12} at {peak_hour:02d}:00              ║
║  Short trips (<1mi): {short_pct}% of all trips                   ║
╚═══════════════════════════════════════════════════════════╝
""")

## ✅ Summary — All 5 Gold Tables Written

| Table | Business question | Key metric |
|---|---|---|
| `daily_revenue` | Revenue per day | + 7-day rolling avg |
| `hourly_patterns` | Demand by hour & day | 168 combinations |
| `payment_analysis` | Payment method breakdown | Tip rate % |
| `top_routes` | Most popular zone pairs | Ranked per zone |
| `distance_bands` | Short vs long journeys | Share % |

---

**Pipeline complete!** You've built a full Medallion Architecture pipeline:

```
TLC Open Data → Bronze (raw) → Silver (clean) → Gold (analytics) → BI
```

**Next steps:**
- Connect a Power BI or Looker Studio dashboard to the Gold Delta tables
- Add Apache Airflow to orchestrate the pipeline on a schedule (Project 3)
- Replace Gold PySpark with dbt models for versioned SQL transformations (Project 2)